# 1. Imports

In [47]:
%pip install qiskit
%pip install qiskit_machine_learning
%pip install qiskit_algorithms


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [48]:
# Data management
import numpy as np
import pandas as pd

# Maths
import math
from math import pi

# Plot
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

# Additional imports
import pylab as pl
from random import random
from numpy import linalg
from qiskit_machine_learning.datasets import ad_hoc_data
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit.circuit.library import ZZFeatureMap
from qiskit.primitives import Sampler
from qiskit_algorithms.state_fidelities import ComputeUncompute

# Plot configuration
%matplotlib inline
sns.set_theme()
sns.set_context("poster")
sns.set_style("ticks")

# 2. Auxiliar functions

## Grover

In [49]:
def get_grover_operator(classical_oracle):
    """
    Entry :
        classical_operator -> A list of booleans or integers indicating which elements are marked.
    Outputs :
        psi -> The initial state of the algorithm.
        G -> The operator coresponding to the Grover algorithm.
    """
    N = len(classical_oracle) # Number of searched elements
    
    psi = np.array([1./np.sqrt(N)]*N) # Initial state
    
    # Reflection operator U
    U = np.array([[i] for i in psi])
    U = 2.*np.dot(U,U.conj().T) - np.eye(N)
    
    # Oracle R
    R = np.eye(N)
    for i in range(len(R)):
        if classical_oracle[i]:
            R[i][i] = -1.
            
    G = np.dot(U,R)
    return psi,G

def success_proba_grover(classical_oracle,m):
    """
    Entries :
        classical oracle -> A list of booleans or integers indicating which elements are marked.
        m -> The number of steps we want to perform
    Outputs :
        steps -> A numpy array listing the steps taken by the algorithm.
        p -> A numpy array with the probability of success of the Grover algorithm for each step in steps.
    Note : The two numpy array have a len of m+1.
    """
    def success_proba(psi):
        """ Return the success probability given the current state psi. """
        s = 0.
        for i in range(len(psi)):
            if classical_oracle[i]:
                s += abs(psi[i])**2
        return s
        
    N = len(classical_oracle)
    
    psi,G = get_grover_operator(classical_oracle) # Grover inital state and operator
    steps = [0]
    p = [success_proba(psi)] # Probability of success
    
    for step in range(1,m+1):
        steps.append(step)
        psi = np.dot(G,psi)
        p.append(success_proba(psi))
        
    return np.array(steps),np.array(p)

In [50]:
def grover(classical_oracle, m):
    """
    Entries :
        classical oracle -> A list of booleans or integers indicating which elements are marked.
        m -> The number of steps we want to perform
    Output :
        Elements measured at the end of the algorithm
    """
    N = len(classical_oracle)

    psi,G = get_grover_operator(classical_oracle) # Grover inital state and operator
    psi = np.dot(np.linalg.matrix_power(G,m),psi) # Final state
    
    p = np.array([np.abs(psi[i])**2 for i in range(N)])
    c = np.random.choice(N,1,p=p) # Measurement
    return c[0]

def grover_opti(classical_oracle, m):
    """
    Entries :
        classical oracle -> A list of booleans or integers indicating which elements are marked.
        m -> The number of steps we want to perform
    Output :
        Elements measured at the end of the algorithm
    """
    N = len(classical_oracle) # Number of elements
    M = sum(classical_oracle) # Number of marked elements
    
    theta = np.arcsin(np.sqrt(M/N))
    f = lambda t:np.sin(2*theta*t+theta)**2 # Theorical probability of success
    
    p = np.array([f(m)/M if classical_oracle[i] else (1-f(m))/(N-M)  for i in range(N)])
    c = np.random.choice(N,1,p=p) # Measurement
    return c[0]

## Search

In [51]:
def classical_search(classical_oracle):
    """
    A classical search of a marked element given an oracle.
    Entry :
         classical_oracle -> The oracle
    Output : {1},{2}
        {1} -> The element found.
        {2} -> The number of necessary steps.
    """
    for i in range(len(classical_oracle)):
        if classical_oracle[i]:
            return i,i+1
    return 0,len(classical_oracle)+1

def quantum_search(classical_oracle, nb_ampli=10, opti=False):
    """
    A quantum search of a marked element given an oracle. Several marked elements aree alowed
    Entries :
         classical_oracle -> The oracle
         nb_ampli -> Number of times we keep repeating the search if it fails.
         opti -> Wether we use the optimized version of Grover.
    Output : {1},{2}
        {1} -> The element found.
        {2} -> The number of necessary steps. This is the number of Grover steps without taking the oracle into consideration. 
    """
    N = len(classical_oracle) # Number of elements
    M = np.ceil(1./np.sin(2*np.arcsin(np.sqrt(1/N)))) # Maximal bound for the number of steps
    m = np.random.randint(0,M+1) # Number of steps is draw randomly
    x = 0 # Element found
    steps = 0 # Number of steps used
    for i in range(nb_ampli): # Each time the search failed (or the first time)
        steps+=m # Adding up the number of steps
        x = grover_opti(classical_oracle,m) if opti else grover(classical_oracle,m) # Use grover to find an element.
        if classical_oracle[x]: # If the search is a success
            break # Stop
        m = np.random.randint(0,M+1) # Else we repeat the search with a new number of steps.
        x = 0
            
    return x,steps

## Activation functions

In [52]:
def linear_activation(x):
    """
    Linear activation function.

    Args:
        x (numpy.ndarray): Input data.

    Returns:
        numpy.ndarray: Output after applying linear function.
    """
    return x

def step_activation(x):
    """
    Step activation function.

    Args:
        x (numpy.ndarray): Input data.

    Returns:
        numpy.ndarray: Output after applying step function.
    """
    return np.where(x > 0, 1, 0)

def sigmoid_activation(x):
    """
    Sigmoid activation function.

    Args:
        x (numpy.ndarray): Input data.

    Returns:
        numpy.ndarray: Output after applying sigmoid function.
    """
    return 1 / (1 + np.exp(-x))

def tanh_activation(x):
    """
    Hyperbolic tangent activation function.

    Args:
        x (numpy.ndarray): Input data.

    Returns:
        numpy.ndarray: Output after applying hyperbolic tangent function.
    """
    return np.tanh(x)

def relu_activation(x):
    """
    Rectified Linear Unit (ReLU) activation function.

    Args:
        x (numpy.ndarray): Input data.

    Returns:
        numpy.ndarray: Output after applying ReLU function.
    """
    return np.maximum(0, x)

def leaky_relu_activation(x, alpha=0.01):
    """
    Leaky ReLU activation function.

    Args:
        x (numpy.ndarray): Input data.
        alpha (float, optional): Slope of the negative part. Defaults to 0.01.

    Returns:
        numpy.ndarray: Output after applying Leaky ReLU function.
    """
    return np.where(x > 0, x, alpha * x)

def softmax_activation(x):
    """
    Softmax activation function.

    Args:
        x (numpy.ndarray): Input data.

    Returns:
        numpy.ndarray: Output after applying softmax function.
    """
    exp_values = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_values / np.sum(exp_values, axis=1, keepdims=True)


## Kernels

In [53]:
adhoc_feature_map = ZZFeatureMap(feature_dimension=2, reps=2, entanglement="linear")
sampler = Sampler()
fidelity = ComputeUncompute(sampler=sampler)
quantum_kernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=adhoc_feature_map)

In [54]:
def sigmoid_kernel(x1, x2, alpha=1, c=0):
    """
    Compute the Sigmoid kernel between two vectors.

    Args:
        x1 (numpy.ndarray): First input vector.
        x2 (numpy.ndarray): Second input vector.
        alpha (float, optional): Scaling parameter. Defaults to 1.
        c (float, optional): Constant term. Defaults to 0.

    Returns:
        float: Sigmoid kernel value.
    """
    return np.tanh(alpha * np.dot(x1, x2) + c)

def laplacian_kernel(x, y, sigma=1.0):
    """
    Compute the Laplacian kernel between two vectors.

    Args:
        x (numpy.ndarray): First input vector.
        y (numpy.ndarray): Second input vector.
        sigma (float, optional): Bandwidth parameter. Defaults to 1.0.

    Returns:
        float: Laplacian kernel value.
    """
    return np.exp(-linalg.norm(x - y) / sigma)

def anova_kernel(x, y, D):
    """
    Compute the Anova kernel between two vectors.

    Args:
        x (numpy.ndarray): First input vector.
        y (numpy.ndarray): Second input vector.
        D (int): Dimensionality of the vectors.

    Returns:
        float: Anova kernel value.
    """
    return sum(np.prod(np.outer(x[:d], y[:d])) for d in range(D + 1))

def chi_squared_kernel(x, y):
    """
    Compute the Chi-squared kernel between two vectors.

    Args:
        x (numpy.ndarray): First input vector.
        y (numpy.ndarray): Second input vector.

    Returns:
        float: Chi-squared kernel value.
    """
    return np.exp(-np.sum((x - y)**2 / (x + y)))

def histogram_intersection_kernel(x, y):
    """
    Compute the Histogram Intersection kernel between two vectors.

    Args:
        x (numpy.ndarray): First input vector.
        y (numpy.ndarray): Second input vector.

    Returns:
        float: Histogram Intersection kernel value.
    """
    return np.sum(np.minimum(x, y))

def linear_kernel(x1, x2):
    """
    Compute the Linear kernel between two vectors.

    Args:
        x1 (numpy.ndarray): First input vector.
        x2 (numpy.ndarray): Second input vector.

    Returns:
        float: Linear kernel value.
    """
    return np.dot(x1, x2)

def polynomial_kernel(x, y, p=3):
    """
    Compute the Polynomial kernel between two vectors.

    Args:
        x (numpy.ndarray): First input vector.
        y (numpy.ndarray): Second input vector.
        p (int, optional): Degree of the polynomial. Defaults to 3.

    Returns:
        float: Polynomial kernel value.
    """
    return (1 + np.dot(x, y)) ** p

def gaussian_kernel(x, y, sigma=5.0):
    """
    Compute the Gaussian (RBF) kernel between two vectors.

    Args:
        x (numpy.ndarray): First input vector.
        y (numpy.ndarray): Second input vector.
        sigma (float, optional): Width of the Gaussian. Defaults to 5.0.

    Returns:
        float: Gaussian kernel value.
    """
    return np.exp(-linalg.norm(x-y)**2 / (2 * (sigma ** 2)))

def rbf_kernel(x, y, gamma=0.1):
    """
    Compute the Radial Basis Function (RBF) kernel between two vectors.

    Args:
        x (numpy.ndarray): First input vector.
        y (numpy.ndarray): Second input vector.
        gamma (float, optional): Inverse of the width of the Gaussian. Defaults to 0.1.

    Returns:
        float: RBF kernel value.
    """
    return np.exp(-gamma * linalg.norm(x - y)**2)


## Testing

In [55]:
def compute_margin(X,y,fit_intercept=False):
    """
    Use SVM of sklearn to compute the margin of a dataset.
    Entries :
        X -> The points
        y -> The classes
        fit_intercept -> If you want to allow the intercept to be fitted. The model coded here haven't this option.
    Outputs :
        w -> The best separator / the most centered one / the on that realizes the margin.
        b -> The intercept
        gamma -> The margin
    """
    y = [1 if i==1 else -1 for i in y]
    clf = LinearSVC(fit_intercept=fit_intercept,max_iter=100000)
    clf.fit(X, y)
    w = clf.coef_[0]
    b=0
    if fit_intercept:
        b = clf.intercept_[0]
    gamma = min([y[i]*(X[i,:].dot(w)+b)/np.linalg.norm(w) for i in range(len(X))])
    w = w/np.linalg.norm(w)
    return w,b,gamma

In [56]:
def accuracy(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)

# 3. Classes

## Perceptron Roget

In [57]:
class Perceptron_Online:
    """
    Perceptron classifier trained using online learning algorithm.

    Parameters:
    -----------
    max_iter : int, optional
        Maximum number of iterations (epochs) for training. Defaults to 10000.
    shuffle : bool, optional
        Whether to shuffle the training dataset between each correction. Defaults to True.
    quantum : bool, optional
        If True, use quantum search for finding misclassified points. Defaults to False.
    nb_ampli : int, optional
        Amplification parameter for quantum search. Defaults to 10.
    opti : bool, optional
        If True, use optimized Grover's algorithm for quantum search. Defaults to False.
    """

    def __init__(self, n_iter=None, shuffle=True, quantum=False, nb_ampli=10, opti=False, gamma=0.01):
        self.shuffle = shuffle  # If we shuffle the training dataset between each correction
        self.coef_ = np.array([0.])  # Default hyperplane
        self.n_iter_ = 0  # Number of iteration (include the number of steps used for the search)
        self.n_correction_ = 0  # Number of correction
        self.quantum = quantum  # If we use the quantum search
        self.nb_ampli = nb_ampli  # The amplification parameter for the quantum search
        self.opti = opti  # If we use the opti Grover
        self.gamma = gamma

    def fit(self, X, y):
        """
        Train the perceptron classifier.

        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training data.
        y : array-like of shape (n_samples,)
            Target labels.

        Returns:
        --------
        int
            The number of iterations.

        Updates the model's coefficient during training.
        """
        max_iter = int(len(X)/self.gamma**2)

        b = True
        self.coef_ = np.array([0.] * len(X[0]))  # Initialization of the coef (can be removed if you want to train successively)
        # Copy of the entry (for the shuffle step)
        X_ = np.array([[j for j in i] for i in X])
        y_ = np.array([1 if i == 1 else -1 for i in y])  # Security to ensure that the classes are {-1,1} and not {0,1}.

        nb = 0
        while b:
            nb += 1
            if nb > max_iter:
                break
            b = False

            oracle = [int(y_[i] * X_[i, :].dot(self.coef_) <= 0) for i in range(len(y_))]  # Oracle for the search

            # The right search (according to the options) is used.
            m, steps = classical_search(oracle) if not self.quantum else quantum_search(oracle, nb_ampli=self.nb_ampli, opti=self.opti)
            self.n_iter_ += steps  # We add the number of steps to the model

            if y_[m] * X_[m, :].dot(self.coef_) <= 0:  # If the search is successful we correct
                self.coef_ = self.coef_ + y_[m] * X_[m, :]
                b = True
                self.n_correction_ += 1
                nb += 1

            if self.shuffle:  # Shuffle
                l = list(range(len(X)))
                np.random.shuffle(l)
                X_ = np.array([X[i] for i in l])
                y_ = np.array([1 if y[i] == 1 else -1 for i in l])
        return self.n_iter_

    def predict(self, X):
        """
        Predict class labels for samples in X.

        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Samples.

        Returns:
        --------
        array-like of shape (n_samples,)
            Predicted class labels.
        """
        return np.array([1 if x.dot(self.coef_) > 0 else -1 for x in X])


In [58]:
class Perceptron_Space:
    """
    Perceptron classifier trained using a set of separators in feature space.

    Parameters:
    -----------
    separators : array-like
        Set of separators.
    nb_ampli : int, optional
        Amplification parameter for quantum search. Defaults to 10.
    quantum : bool, optional
        If True, use quantum search for finding the best separator. Defaults to False.
    opti : bool, optional
        If True, use optimized Grover's algorithm for quantum search. Defaults to False.
    """

    def __init__(self, nb_hyperplanes, nb_ampli=10, quantum=False, opti=False):
        self.separators = None  # Set of separators
        self.nb_hyperplanes = nb_hyperplanes
        self.selected = 0  # The selected separator
        self.n_iter_ = 0  # Number of steps
        self.coef_ = None  # Default hyperplane
        self.quantum = quantum  # If we use the quantum search
        self.nb_ampli = nb_ampli  # The amplification parameter for the quantum search
        self.opti = opti  # If we use the opti Grover

    def fit(self, X, y):
        """
        Train the perceptron classifier.

        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training data.
        y : array-like of shape (n_samples,)
            Target labels.

        Returns:
        --------
        int
            The number of iterations.

        One of the "separators" will be chosen.
        """
        if not self.separators:
            len_data = len(X[0])
            self.separators = np.random.multivariate_normal([0]*len_data,np.eye(len_data),size=self.nb_hyperplanes)
            np.random.shuffle(self.separators)
            self.coef_ = self.separators[self.selected]
            
        X_ = np.array([[j for j in i] for i in X])
        y_ = np.array([1 if i == 1 else -1 for i in y])  # Security to ensure that the classes are {-1,1} and not {0,1}.

        # Oracle over the hyerplanes
        oracle = [int(all([y_[i] * X_[i, :].dot(self.separators[k]) > 0 for i in range(len(X_))])) for k in range(len(self.separators))]

        # Search
        k, steps = classical_search(oracle) if not self.quantum else quantum_search(oracle, nb_ampli=self.nb_ampli, opti=self.opti)

        self.n_iter_ += steps * len(X_)  # The number of steps is multiplied by the comlexity of the oracle.
        if all([y_[i] * X_[i, :].dot(self.separators[k]) > 0 for i in range(len(X_))]):  # If the search is successful we take the hyperplane.
            self.selected = k
            self.coef_ = self.separators[self.selected]

        return self.n_iter_

    def predict(self, X):
        """
        Predict class labels for samples in X.

        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Samples.

        Returns:
        --------
        array-like of shape (n_samples,)
            Predicted class labels.
        """
        if self.separators is not None:
            len_data = len(X[0])
            self.separators = np.random.multivariate_normal([0]*len_data,np.eye(len_data),size=self.nb_hyperplanes)
            np.random.shuffle(self.separators)
            self.coef_ = self.separators[self.selected]
        return np.array([1 if x.dot(self.separators[self.selected]) > 0 else -1 for x in X])


In [59]:
class Perceptron_Hybrid:
    """
    Perceptron classifier trained using a hybrid approach of classical and quantum search.

    Parameters:
    -----------
    separators : array-like
        Set of separators.
    nb_ampli : int, optional
        Amplification parameter for quantum search. Defaults to 10.
    quantum : bool, optional
        If True, use quantum search for finding the best separator. Defaults to False.
    opti : bool, optional
        If True, use optimized Grover's algorithm for quantum search. Defaults to False.
    """

    def __init__(self, nb_hyperplanes, nb_ampli=10, quantum=False, opti=False):
        self.separators = None  # Set of separators
        self.nb_hyperplanes = nb_hyperplanes
        self.selected = 0  # The selected separator
        self.n_iter_ = 0  # Number of steps
        self.coef_ = None  # Default hyperplane
        self.quantum = quantum  # If we use the quantum search
        self.nb_ampli = nb_ampli  # The amplification parameter for the quantum search
        self.opti = opti  # If we use the opti Grover

    def fit(self, X, y):
        """
        Train the perceptron classifier.

        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training data.
        y : array-like of shape (n_samples,)
            Target labels.

        Returns:
        --------
        int
            The number of iterations.

        One of the "separators" will be chosen.
        """
        if not self.separators:
            len_data = len(X[0])
            self.separators = np.random.multivariate_normal([0]*len_data,np.eye(len_data),size=self.nb_hyperplanes)
            np.random.shuffle(self.separators)
            self.coef_ = self.separators[self.selected]
            
        X_ = np.array([[j for j in i] for i in X])
        y_ = np.array([1 if i == 1 else -1 for i in y])  # Security to ensure that the classes are {-1,1} and not {0,1}.

        for k in range(len(self.separators)):  # For each separator
            # Oracle over the points
            oracle = [int(y_[i] * X_[i, :].dot(self.separators[k]) <= 0) for i in range(len(y_))]
            # Search
            m, steps = classical_search(oracle) if not self.quantum else quantum_search(oracle, nb_ampli=self.nb_ampli, opti=self.opti)

            self.n_iter_ += steps

            if y_[m] * X_[m, :].dot(self.separators[k]) > 0:  # If successful we choose this hyperplane
                self.selected = k
                self.coef_ = self.separators[self.selected]
                break

        return self.n_iter_

    def predict(self, X):
        """
        Predict class labels for samples in X.

        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Samples.

        Returns:
        --------
        array-like of shape (n_samples,)
            Predicted class labels.
        """
        if self.separators is not None:
            len_data = len(X[0])
            self.separators = np.random.multivariate_normal([0]*len_data,np.eye(len_data),size=self.nb_hyperplanes)
            np.random.shuffle(self.separators)
            self.coef_ = self.separators[self.selected]
        return np.array([1 if x.dot(self.separators[self.selected]) > 0 else -1 for x in X])


## Perceptron Albert

In [60]:
class Perceptron(object):

    def __init__(self, n_iter=1):
        self.n_iter = n_iter

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features, dtype=np.float64)
        self.b = 0.0

        for t in range(self.n_iter):
            for i in range(n_samples):
                if self.predict(X[i])[0] != y[i]:
                    self.w += y[i] * X[i]
                    self.b += y[i]

    def project(self, X):
        return np.dot(X, self.w) + self.b

    def predict(self, X):
        X = np.atleast_2d(X)
        return np.sign(self.project(X))

In [61]:
class KernelPerceptron(object):

    def __init__(self, kernel, n_iter=1):
        self.kernel = kernel
        self.n_iter = n_iter

    def fit(self, X, y):
        n_samples, _ = X.shape
        self.alpha = np.zeros(n_samples, dtype=np.float64)

        # Gram matrix
        K = np.zeros((n_samples, n_samples))
        for i in range(n_samples):
            for j in range(n_samples):
                K[i,j] = self.kernel(X[i], X[j])

        for _ in range(self.n_iter):
            for i in range(n_samples):
                if np.sign(np.sum(K[:,i] * self.alpha * y)) != y[i]:
                    self.alpha[i] += 1.0

        # Support vectors
        sv = self.alpha > 1e-5
        self.alpha = self.alpha[sv]
        self.sv = X[sv]
        self.sv_y = y[sv]
        print(f"{len(self.alpha)} support vectors out of {n_samples} points")

    def project(self, X):
        y_predict = np.zeros(len(X))
        for i in range(len(X)):
            s = 0
            for a, sv_y, sv in zip(self.alpha, self.sv_y, self.sv):
                s += a * sv_y * self.kernel(X[i], sv)
            y_predict[i] = s
        return y_predict

    def predict(self, X):
        X = np.atleast_2d(X)
        return np.sign(self.project(X))

## SVM

In [62]:
class SVM:

    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iter=1000):
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.n_iter = n_iter
        self.w = None
        self.b = None

    def fit(self, X, y):
        n_samples, n_features = X.shape

        y_ = np.where(y <= 0, -1, 1)

        # init weights
        self.w = np.zeros(n_features)
        self.b = 0

        for _ in range(self.n_iter):
            for idx, x_i in enumerate(X):
                condition = y_[idx] * (np.dot(x_i, self.w) - self.b) >= 1
                if condition:
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    self.w -= self.lr * (2 * self.lambda_param * self.w - np.dot(x_i, y_[idx]))
                    self.b -= self.lr * y_[idx]


    def predict(self, X):
        approx = np.dot(X, self.w) - self.b
        return np.sign(approx)

## MLP

In [63]:
class MLP(object):
    """A Multilayer Perceptron class.
    """

    def __init__(self, num_inputs=3, hidden_layers=[3, 3], num_outputs=2):
        """Constructor for the MLP. Takes the number of inputs,
            a variable number of hidden layers, and number of outputs

        Args:
            num_inputs (int): Number of inputs
            hidden_layers (list): A list of ints for the hidden layers
            num_outputs (int): Number of outputs
        """

        self.num_inputs = num_inputs
        self.hidden_layers = hidden_layers
        self.num_outputs = num_outputs

        # create a generic representation of the layers
        layers = [num_inputs] + hidden_layers + [num_outputs]

        # create random connection weights for the layers
        weights = []
        for i in range(len(layers) - 1):
            w = np.random.rand(layers[i], layers[i + 1])
            weights.append(w)
        self.weights = weights

        # save derivatives per layer
        derivatives = []
        for i in range(len(layers) - 1):
            d = np.zeros((layers[i], layers[i + 1]))
            derivatives.append(d)
        self.derivatives = derivatives

        # save activations per layer
        activations = []
        for i in range(len(layers)):
            a = np.zeros(layers[i])
            activations.append(a)
        self.activations = activations


    def forward_propagate(self, inputs):
        """Computes forward propagation of the network based on input signals.

        Args:
            inputs (ndarray): Input signals
        Returns:
            activations (ndarray): Output values
        """

        # the input layer activation is just the input itself
        activations = inputs

        # save the activations for backpropogation
        self.activations[0] = activations

        # iterate through the network layers
        for i, w in enumerate(self.weights):
            # calculate matrix multiplication between previous activation and weight matrix
            net_inputs = np.dot(activations, w)

            # apply sigmoid activation function
            activations = self._sigmoid(net_inputs)

            # save the activations for backpropogation
            self.activations[i + 1] = activations

        # return output layer activation
        return activations


    def back_propagate(self, error):
        """Backpropogates an error signal.
        Args:
            error (ndarray): The error to backprop.
        Returns:
            error (ndarray): The final error of the input
        """

        # iterate backwards through the network layers
        for i in reversed(range(len(self.derivatives))):

            # get activation for previous layer
            activations = self.activations[i+1]

            # apply sigmoid derivative function
            delta = error * self._sigmoid_derivative(activations)

            # reshape delta as to have it as a 2d array
            delta_re = delta.reshape(delta.shape[0], -1).T

            # get activations for current layer
            current_activations = self.activations[i]

            # reshape activations as to have them as a 2d column matrix
            current_activations = current_activations.reshape(current_activations.shape[0],-1)

            # save derivative after applying matrix multiplication
            self.derivatives[i] = np.dot(current_activations, delta_re)

            # backpropogate the next error
            error = np.dot(delta, self.weights[i].T)


    def train(self, inputs, targets, epochs, learning_rate):
        """Trains model running forward prop and backprop
        Args:
            inputs (ndarray): X
            targets (ndarray): Y
            epochs (int): Num. epochs we want to train the network for
            learning_rate (float): Step to apply to gradient descent
        """
        # now enter the training loop
        for i in range(epochs):
            sum_errors = 0

            # iterate through all the training data
            for j, input in enumerate(inputs):
                target = targets[j]

                # activate the network!
                output = self.forward_propagate(input)

                error = target - output

                self.back_propagate(error)

                # now perform gradient descent on the derivatives
                # (this will update the weights
                self.gradient_descent(learning_rate)

                # keep track of the MSE for reporting later
                sum_errors += self._mse(target, output)

            # Epoch complete, report the training error
            print("Error: {} at epoch {}".format(sum_errors / len(items), i+1))

        print("Training complete!")
        print("=====")


    def gradient_descent(self, learningRate=1):
        """Learns by descending the gradient
        Args:
            learningRate (float): How fast to learn.
        """
        # update the weights by stepping down the gradient
        for i in range(len(self.weights)):
            weights = self.weights[i]
            derivatives = self.derivatives[i]
            weights += derivatives * learningRate


    def _sigmoid(self, x):
        """Sigmoid activation function
        Args:
            x (float): Value to be processed
        Returns:
            y (float): Output
        """

        y = 1.0 / (1 + np.exp(-x))
        return y


    def _sigmoid_derivative(self, x):
        """Sigmoid derivative function
        Args:
            x (float): Value to be processed
        Returns:
            y (float): Output
        """
        return x * (1.0 - x)


    def _mse(self, target, output):
        """Mean Squared Error loss function
        Args:
            target (ndarray): The ground trut
            output (ndarray): The predicted values
        Returns:
            (float): Output
        """
        return np.average((target - output) ** 2)

## MLP Adjusted

In [64]:
class MLP(object):
    """A Multilayer Perceptron class.
    """

    def __init__(self, num_inputs=3, hidden_layers=[3, 3], num_outputs=2):
        """Constructor for the MLP. Takes the number of inputs,
            a variable number of hidden layers, and number of outputs

        Args:
            num_inputs (int): Number of inputs
            hidden_layers (list): A list of ints for the hidden layers
            num_outputs (int): Number of outputs
        """

        self.num_inputs = num_inputs
        self.hidden_layers = hidden_layers
        self.num_outputs = num_outputs

        # create a generic representation of the layers
        layers = [num_inputs] + hidden_layers + [num_outputs]

        # create random connection weights for the layers
        weights = []
        for i in range(len(layers) - 1):
            w = np.random.rand(layers[i], layers[i + 1])
            weights.append(w)
        self.weights = weights

        # save derivatives per layer
        derivatives = []
        for i in range(len(layers) - 1):
            d = np.zeros((layers[i], layers[i + 1]))
            derivatives.append(d)
        self.derivatives = derivatives

        # save activations per layer
        activations = []
        for i in range(len(layers)):
            a = np.zeros(layers[i])
            activations.append(a)
        self.activations = activations


    def forward_propagate(self, inputs):
        """Computes forward propagation of the network based on input signals.

        Args:
            inputs (ndarray): Input signals
        Returns:
            activations (ndarray): Output values
        """

        # the input layer activation is just the input itself
        activations = inputs

        # save the activations for backpropogation
        self.activations[0] = activations

        # iterate through the network layers
        for i, w in enumerate(self.weights):
            # calculate matrix multiplication between previous activation and weight matrix
            net_inputs = np.dot(activations, w)

            # apply sigmoid activation function
            activations = self._sigmoid(net_inputs)

            # save the activations for backpropogation
            self.activations[i + 1] = activations

        # return output layer activation
        return activations


    def back_propagate(self, error):
        """Backpropogates an error signal.
        Args:
            error (ndarray): The error to backprop.
        Returns:
            error (ndarray): The final error of the input
        """

        # iterate backwards through the network layers
        for i in reversed(range(len(self.derivatives))):

            # get activation for previous layer
            activations = self.activations[i+1]

            # apply sigmoid derivative function
            delta = error * self._sigmoid_derivative(activations)

            # reshape delta as to have it as a 2d array
            delta_re = delta.reshape(delta.shape[0], -1).T

            # get activations for current layer
            current_activations = self.activations[i]

            # reshape activations as to have them as a 2d column matrix
            current_activations = current_activations.reshape(current_activations.shape[0],-1)

            # save derivative after applying matrix multiplication
            self.derivatives[i] = np.dot(current_activations, delta_re)

            # backpropogate the next error
            error = np.dot(delta, self.weights[i].T)


    def train(self, inputs, targets, epochs, learning_rate):
        """Trains model running forward prop and backprop
        Args:
            inputs (ndarray): X
            targets (ndarray): Y
            epochs (int): Num. epochs we want to train the network for
            learning_rate (float): Step to apply to gradient descent
        """
        # now enter the training loop
        for i in range(epochs):
            sum_errors = 0

            # iterate through all the training data
            for j, input in enumerate(inputs):
                target = targets[j]

                # activate the network!
                output = self.forward_propagate(input)

                error = target - output

                self.back_propagate(error)

                # now perform gradient descent on the derivatives
                # (this will update the weights
                self.gradient_descent(learning_rate)

                # keep track of the MSE for reporting later
                sum_errors += self._mse(target, output)

            # Epoch complete, report the training error
            print("Error: {} at epoch {}".format(sum_errors / len(items), i+1))

        print("Training complete!")
        print("=====")


    def gradient_descent(self, learningRate=1):
        """Learns by descending the gradient
        Args:
            learningRate (float): How fast to learn.
        """
        # update the weights by stepping down the gradient
        for i in range(len(self.weights)):
            weights = self.weights[i]
            derivatives = self.derivatives[i]
            weights += derivatives * learningRate


    def _sigmoid(self, x):
        """Sigmoid activation function
        Args:
            x (float): Value to be processed
        Returns:
            y (float): Output
        """

        y = 1.0 / (1 + np.exp(-x))
        return y


    def _sigmoid_derivative(self, x):
        """Sigmoid derivative function
        Args:
            x (float): Value to be processed
        Returns:
            y (float): Output
        """
        return x * (1.0 - x)


    def _mse(self, target, output):
        """Mean Squared Error loss function
        Args:
            target (ndarray): The ground trut
            output (ndarray): The predicted values
        Returns:
            (float): Output
        """
        return np.average((target - output) ** 2)

# 4. Classes instances

## 4.1 Function list

In [65]:
kernel_list = [
    quantum_kernel,
    rbf_kernel,
    sigmoid_kernel,
    laplacian_kernel,
    anova_kernel,
    chi_squared_kernel,
    histogram_intersection_kernel,
    linear_kernel,
    polynomial_kernel,
    gaussian_kernel,
    rbf_kernel,
]

In [66]:
activation_list = [
    linear_activation,
    step_activation,
    sigmoid_activation,
    tanh_activation,
    relu_activation,
    leaky_relu_activation,
    softmax_activation,
]

## 4.2 Hyperparameters

In [67]:
learning_rate = 0.001
n_iter = 1000
kernel = quantum_kernel

# Parameters
lambda_param = 0.01
gamma = 0.01
eps = 0.01

# Roget
nb_hyperplanes = math.ceil(np.log(eps)/np.log(1-np.sqrt(2/pi)*gamma))
nb_ampli_online = math.ceil(np.log(eps*gamma**2)/np.log(3/4))
nb_ampli_space = math.ceil(np.log(eps)/np.log(3/4))
nb_ampli_hybrid = math.ceil(np.log(1-(1-eps/2)**(1/(nb_hyperplanes-1)))/np.log(3/4))
#nb_hyperplanes = math.ceil(math.log(eps)/math.log(1-math.sqrt(2/pi)*gamma))

## 4.3 Classifiers

In [68]:
# Classical online perceptron
pt_co_roget = Perceptron_Online(shuffle=False)

In [69]:
# Roget Online quantum perceptron
pt_qo_roget = Perceptron_Online(nb_ampli=nb_ampli_online, shuffle=False, quantum=True)

In [70]:
# Roget Version space classical perceptron
pt_cvs_roget = Perceptron_Space(nb_hyperplanes=nb_hyperplanes)

In [71]:
# Roget Version space quantum perceptron
pt_qvs_roget = Perceptron_Space(nb_hyperplanes=nb_hyperplanes, nb_ampli=nb_ampli_space, quantum=True, opti=True) # Optimization here

In [72]:
# Roget Hybrid quantum perceptron
pt_qh_roget = Perceptron_Hybrid(nb_hyperplanes=nb_hyperplanes, nb_ampli=nb_ampli_hybrid, quantum=True)

In [73]:
# Albert classical perceptron
pt_albert = Perceptron(n_iter=n_iter)

In [74]:
# Albert Kernel perceptron
kpt_albert = KernelPerceptron(kernel, n_iter=n_iter)

In [75]:
# Albert SVM
svm = SVM(learning_rate=learning_rate, lambda_param=lambda_param, n_iter=n_iter)

In [76]:
# Albert MLP
mlp = None

In [77]:
# Roget MLP Classical online perceptron
mlp_pt_co_roget = None

In [78]:
# Roget MLP 
mlp_pt_qo_roget = None

In [79]:
# Roget MLP 
mlp_pt_cvs_roget = None

In [80]:
# Roget MLP 
mlp_pt_qvs_roget = None

In [81]:
# Roget MLP 
mlp_pt_qh_roget = None

# 5. Datasets

In [82]:
def get_iris():
    """
    Load Iris with some restrictions.
    """
    iris = datasets.load_iris()
    X_ = iris.data[:,:2]
    Y_ = iris.target
    X = np.array([X_[i]+np.array([0,3.3]) for i in range(len(X_)) if Y_[i] != 2])
    y = np.array([1 if i==0 else -1 for i in Y_ if i != 2])
    size_max = max([np.sqrt(sum(x**2)) for x in X])
    X = np.array([[i/size_max for i in x] for x in X])
    
    u = sum(X)/len(X)
    X = X-u + np.array([0.013,0.])
    
    return X,y

In [83]:
def get_hard(N=200):
    X = []
    for i in range(0,N):
        X.append([(0)**i if j<i else ((-1)**(i+1) if i==j else 0) for j in range(N)])
    y = np.array([1 if i%2==0 else -1 for i in range(N)])
    X = np.array(X)
    return X,y

In [84]:
dataset_list = [
    get_iris,
    get_hard
]

# 6. Testing

In [85]:
clf_list = [
    pt_co_roget,
    pt_qo_roget,
    pt_cvs_roget,
    pt_qvs_roget,
    pt_qh_roget,
    pt_albert,
    kpt_albert,
    svm,
    mlp_pt_co_roget,
    mlp_pt_qo_roget,
    mlp_pt_cvs_roget,
    mlp_pt_qvs_roget,
    mlp_pt_qh_roget,
]

In [86]:
train_size = 0.66

X, y = get_iris()
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=train_size)

_, _, gamma = compute_margin(X,y,fit_intercept=False)

c:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


In [87]:
for clf in clf_list:
    clf_class = type(clf)
    clf.fit(X_train,y_train)
    predictions = clf.predict(X_test)
    print(f"{clf_class} classification accuracy: {accuracy(y_test, predictions)}")

<class '__main__.Perceptron_Online'> classification accuracy: 1.0
<class '__main__.Perceptron_Online'> classification accuracy: 1.0


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

# 7. Visualization functions

In [ ]:
def visualize_svm(clf):
    def get_hyperplane_value(x, w, b, offset):
        return (-w[0] * x + b + offset) / w[1]

    fig = plt.figure()
    ax = fig.add_subplot(1, 1, 1)
    plt.scatter(X[:, 0], X[:, 1], marker="o", c=y)

    x0_1 = np.amin(X[:, 0])
    x0_2 = np.amax(X[:, 0])

    x1_1 = get_hyperplane_value(x0_1, clf.w, clf.b, 0)
    x1_2 = get_hyperplane_value(x0_2, clf.w, clf.b, 0)

    x1_1_m = get_hyperplane_value(x0_1, clf.w, clf.b, -1)
    x1_2_m = get_hyperplane_value(x0_2, clf.w, clf.b, -1)

    x1_1_p = get_hyperplane_value(x0_1, clf.w, clf.b, 1)
    x1_2_p = get_hyperplane_value(x0_2, clf.w, clf.b, 1)

    ax.plot([x0_1, x0_2], [x1_1, x1_2], "y--")
    ax.plot([x0_1, x0_2], [x1_1_m, x1_2_m], "k")
    ax.plot([x0_1, x0_2], [x1_1_p, x1_2_p], "k")

    x1_min = np.amin(X[:, 1])
    x1_max = np.amax(X[:, 1])
    ax.set_ylim([x1_min - 3, x1_max + 3])

    plt.show()


# 8. Visualization results